# RunResult, Streaming & Max Turns

Prior notebooks used `Runner.run`/`run_sync`/`run_streamed` at a surface
level. This notebook goes deep on what a run actually returns and how far
it can go: `RunResult`, streaming events one level deeper, and `max_turns`.


## `RunResult` in depth

`final_output` is only one field: `new_items`, `raw_responses`, `last_agent`, and
`interruptions` are also worth knowing.


In [5]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo: runs one agent call and shows every field of RunResult, not just final_output
async def demo_run_result() -> None:
    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        instructions="test",
    )

    # Run the Agent
    result = await Runner.run(
        agent,
        "What is Kage-level?",
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    [
                        assistant_message("Kage-level is the highest ninja rank."),
                    ]
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e05-runresult-demo",
        ),
    )

    print(
        "[final_output]", result.final_output
    )  # the agent's final answer: text by default, or an object of output_type if you set one
    print(
        "[last_agent]", result.last_agent.name
    )  # which Agent made final_output. After a handoff it is not the starting agent
    print(
        "[new_items]", [type(item).__name__ for item in result.new_items]
    )  # every item made in this run: messages, tool calls, handoffs, and so on
    print(
        "[raw_responses count]", len(result.raw_responses)
    )  # one response for each model call in this run. The list grows with each turn or tool trip


await demo_run_result()
# Outside a notebook: asyncio.run(demo_run_result())

[final_output] Kage-level is the highest ninja rank.
[last_agent] Leaf Village Assistant
[new_items] ['MessageOutputItem']
[raw_responses count] 1


## Streaming events, one level deeper

`run_item_stream_event` items carry an `item` with its own `.type` — this is how
you tell a "message produced" event apart from a "tool called" event.


In [6]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
    function_tool,  # turns a normal function into a tool an Agent can call
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# @function_tool callback: gives the streaming agent something to call during the run, so the
# event stream below has a tool_call_item and a tool_call_output_item to show, and not only a message
@function_tool
def get_ninja_rank(name: str) -> str:
    """Look up a ninja's rank (stubbed for this demo)."""
    return "Kage-level"


# Demo: streams one run event by event and prints each event's type, including the tool call in the middle
async def demo_streaming() -> None:
    # Initialize the agent
    streaming_agent = Agent(
        name="Leaf Village Assistant",
        instructions="test",
        # tools: the tools this agent is allowed to call
        tools=[get_ninja_rank],
    )

    # Run the Agent
    streamed = Runner.run_streamed(
        streaming_agent,
        "What rank is Naruto Uzumaki?",
        # run_config: settings for just this run
        run_config=RunConfig(
            # model: use this model for this run only
            model=ScriptedModel(
                [
                    [function_call("get_ninja_rank", {"name": "Naruto Uzumaki"}, call_id="c1")],
                    [assistant_message("Naruto Uzumaki is Kage-level.")],
                ]
            ),
            # workflow_name: the name this run shows up under in tracing
            workflow_name="e05-stream-demo",
        ),
    )

    # run_item_stream_event is the one event type that holds a finished RunItem (a
    # tool call, its result, or a message). Its item.type tells them apart, so it
    # is handled on its own. Other event types (turn or agent events,
    # raw model pieces) have no item, so we just print their
    # own .type
    async for event in streamed.stream_events():
        if event.type == "run_item_stream_event":
            print(f"[run_item_stream_event] item.type={event.item.type}")
        else:
            print(f"[{event.type}]")


await demo_streaming()
# Outside a notebook: asyncio.run(demo_streaming())

[agent_updated_stream_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[run_item_stream_event] item.type=tool_call_item
[run_item_stream_event] item.type=tool_call_output_item
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[run_item_stream_event] item.type=message_output_item


## `max_turns` and `MaxTurnsExceeded`

Script a model that calls a tool forever; with `max_turns=2` the run gives up
instead of looping indefinitely.


In [8]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    MaxTurnsExceeded,  # error raised when a run uses more than max_turns model calls
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
    function_tool,  # turns a normal function into a tool an Agent can call
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# @function_tool callback: simple and always works. The model keeps calling it
# again and again below, which pushes the run past max_turns
@function_tool
def ping() -> str:
    """Always returns pong — used here to create an endless tool-call loop."""
    return "pong"


# Demo: stops a tool-calling loop that never ends with max_turns and catches the MaxTurnsExceeded error
async def demo_max_turns() -> None:
    # Initialize the agent
    loop_agent = Agent(
        name="loop-demo",
        instructions="test",
        # tools: the tools this agent is allowed to call
        tools=[ping],
    )
    infinite_steps = [
        [
            function_call("ping", {}, call_id=f"c{i}"),
        ]
        for i in range(10)
    ]

    # Run the Agent
    try:
        await Runner.run(
            loop_agent,
            "loop",
            # max_turns: the most model calls allowed before the run gives up
            max_turns=5,
            # run_config: settings for just this run
            run_config=RunConfig(
                # model: use this model for this run only
                model=ScriptedModel(infinite_steps),
                # workflow_name: the name this run shows up under in tracing
                workflow_name="e05-max-turns",
            ),
        )
    except MaxTurnsExceeded as error:
        print("[expected]", error)


await demo_max_turns()
# Outside a notebook: asyncio.run(demo_max_turns())

[expected] Max turns (5) exceeded


## Run it live — optional, needs a real key

One real call that exercises all three things this notebook covered: `run_streamed`
for streaming, a tool call so `new_items`/`raw_responses` have more than one entry to
inspect (the `RunResult` deep dive), and an explicit `max_turns` cap.


In [9]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
    function_tool,  # turns a normal function into a tool an Agent can call
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# @function_tool callback: the tool the live agent really calls with the real API
@function_tool
def get_ninja_rank(name: str) -> str:
    """Look up a ninja's rank (stubbed for this demo)."""
    return "Kage-level"


# Demo: one real API call that uses streaming, a real tool trip, and an explicit max_turns limit
async def demo_live() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the agent
    live_agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly. Use the get_ninja_rank tool to look up any ninja's rank.",
        # tools: the tools this agent is allowed to call
        tools=[get_ninja_rank],
    )

    # Run the Agent — run_streamed here, so the same real API call is read as a stream of events
    streamed = Runner.run_streamed(
        live_agent,
        "What rank is Naruto Uzumaki?",
        # max_turns: the most model calls allowed before the run gives up
        max_turns=5,
        # run_config: settings for just this run
        run_config=RunConfig(workflow_name="e05-live-demo"),
    )

    # The same run_item_stream_event and item.type split as demo_streaming above, but now with
    # the real API's events and not scripted ones
    async for event in streamed.stream_events():
        if event.type == "run_item_stream_event":
            print(f"[run_item_stream_event] item.type={event.item.type}")
        else:
            print(f"[{event.type}]")

    # When streaming ends, RunResultStreaming has the same RunResult fields
    print("-" * 50)
    print(
        "[final_output]", streamed.final_output
    )  # the agent's final answer, now made after a real tool call trip
    print("[last_agent]", streamed.last_agent.name)  # which Agent made final_output
    print(
        "[new_items]", [type(item).__name__ for item in streamed.new_items]
    )  # this time it has tool_call and tool_call_output items, and not only a message
    print(
        "[raw_responses count]", len(streamed.raw_responses)
    )  # more than 1 here: one model call asks for the tool, and one more answers using its result


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

[agent_updated_stream_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[run_item_stream_event] item.type=tool_call_item
[run_item_stream_event] item.type=tool_call_output_item
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[raw_response_event]
[run_item_stream_event] item.type=message_output_item
--------------------------------------------------
[final_output] Naruto Uzum

## Summary

- `RunResult` carries far more than `final_output`: `new_items`,
  `raw_responses`, `last_agent`, `interruptions`.
- A `run_item_stream_event`'s `item.type` (e.g. `tool_call_item`,
  `message_output_item`) is how you tell streamed events apart.
- `max_turns` caps model calls per run; past it, `Runner.run` raises
  `MaxTurnsExceeded` instead of looping indefinitely.
